# Spaceship Titanic with IMBD Helper

This notebook keeps the intended flow explicit:

`DataChecker -> RFE FeatureSelector -> all L1 model families -> KFoldPredictor`

In [1]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from tqdm import tqdm
from xgboost import XGBClassifier

from Data_checker import DataChecker
from Feature_selector import FeatureSelector
from Kfold_predictor import KFoldPredictor

In [2]:
DATA_DIR = Path("demo_datasets/spaceship-titanic")
OUTPUT_DIR = Path("outputs/spaceship-titanic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Transported"
RANDOM_STATE = 42

# This is the full helper flow. Lower these only when you want a smoke run.
N_SPLITS = 3
N_REPEATS = 5
RUN_RFE = True
RFE_CV = 3
RFE_FEATURE_COUNTS = None  # None means try 1..n_features.

In [3]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train_df.shape)
print("test:", test_df.shape)
train_df.head()

train: (8693, 14)
test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
SPEND_COLUMNS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def split_column(df, column, sep, width, fill_value="Missing"):
    parts = df[column].fillna(fill_value).astype(str).str.split(sep, expand=True)
    return parts.reindex(columns=range(width), fill_value=fill_value)


def engineer_spaceship_features(df):
    data = df.copy()

    passenger_parts = split_column(data, "PassengerId", "_", 2, "0")
    data["PassengerGroup"] = pd.to_numeric(passenger_parts[0], errors="coerce")
    data["PassengerNumber"] = pd.to_numeric(passenger_parts[1], errors="coerce")

    cabin_parts = split_column(data, "Cabin", "/", 3)
    data["CabinDeck"] = cabin_parts[0]
    data["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    data["CabinSide"] = cabin_parts[2]

    data["LastName"] = (
        data["Name"]
        .fillna("Unknown Unknown")
        .astype(str)
        .str.split()
        .str[-1]
        .fillna("Unknown")
    )

    for column in SPEND_COLUMNS:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data["TotalSpend"] = data[SPEND_COLUMNS].fillna(0).sum(axis=1)
    data["NoSpend"] = (data["TotalSpend"] == 0).astype(int)
    data["SpendPerAge"] = data["TotalSpend"] / (
        pd.to_numeric(data["Age"], errors="coerce").fillna(0) + 1
    )

    return data.drop(columns=["PassengerId", "Cabin", "Name"], errors="ignore")

In [5]:
y = train_df[TARGET].astype(int)
X = engineer_spaceship_features(train_df.drop(columns=[TARGET]))
X_test = engineer_spaceship_features(test_df).reindex(columns=X.columns)

categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide",
    "LastName",
]
categorical_features = [col for col in categorical_features if col in X.columns]
numeric_features = [col for col in X.columns if col not in categorical_features]

X[numeric_features] = X[numeric_features].apply(pd.to_numeric, errors="coerce")
X_test[numeric_features] = X_test[numeric_features].apply(pd.to_numeric, errors="coerce")

medians = X[numeric_features].median(numeric_only=True).fillna(0)
X[numeric_features] = X[numeric_features].fillna(medians)
X_test[numeric_features] = X_test[numeric_features].fillna(medians)

X[categorical_features] = X[categorical_features].fillna("Missing").astype(str)
X_test[categorical_features] = X_test[categorical_features].fillna("Missing").astype(str)

# DataChecker convention: 0 = categorical, 1 = numerical.
typeofFeatures_raw = [0 if col in categorical_features else 1 for col in X.columns]

print("X shape:", X.shape)
print("Categorical features:", categorical_features)
print("Numerical features:", numeric_features)
X.head()

X shape: (8693, 19)
Categorical features: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide', 'LastName']
Numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'PassengerGroup', 'PassengerNumber', 'CabinNum', 'TotalSpend', 'NoSpend', 'SpendPerAge']


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,PassengerGroup,PassengerNumber,CabinDeck,CabinNum,CabinSide,LastName,TotalSpend,NoSpend,SpendPerAge
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,1,1,B,0.0,P,Ofracculy,0.0,1,0.000000
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,2,1,F,0.0,S,Vines,736.0,0,29.440000
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,3,1,A,0.0,S,Susent,10383.0,0,175.983051
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,3,2,A,0.0,S,Susent,5176.0,0,152.235294
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,4,1,F,1.0,S,Santantines,1091.0,0,64.176471


In [6]:
# First DataChecker pass: encode text columns exactly once, then run RFE on numeric data.
raw_checker = DataChecker(
    X=X,
    y=y,
    X_test=X_test,
    mode="class",
    typeofFeatures=typeofFeatures_raw,
)
raw_checker.varify_data_types()
raw_checker.apply_transformations(use_target_encoder=False)

X_encoded = raw_checker.X.copy()
X_test_encoded = raw_checker.X_test.copy()
type_map = dict(zip(X_encoded.columns, typeofFeatures_raw))

Auto-detected text/object columns: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide', 'LastName']
Encoded 7 text columns with OrdinalEncoder.
Categorical features: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide', 'LastName']
Numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'PassengerGroup', 'PassengerNumber', 'CabinNum', 'TotalSpend', 'NoSpend', 'SpendPerAge']
No numerical features found, skipping numerical transformations.
No categorical features found, skipping target encoding.


In [7]:
if RUN_RFE:
    rfe_candidates = RFE_FEATURE_COUNTS or list(range(1, X_encoded.shape[1] + 1))
    feature_selector = FeatureSelector(
        X=X_encoded,
        y=y,
        mode="class",
        TypesofFeatures=rfe_candidates,
        use_gpu=False,
        cv=RFE_CV,
        min_selected_features=1,
    )
    baseline_scores = feature_selector.get_baseline()
    rfe_scores = feature_selector.get_scores_with_different_thresholds()
    X_model, fitted_selector = feature_selector.get_new_dataset()
    X_test_model = pd.DataFrame(
        fitted_selector.transform(X_test_encoded),
        columns=X_model.columns,
        index=X_test_encoded.index,
    )
else:
    baseline_scores = None
    rfe_scores = None
    X_model = X_encoded.copy()
    X_test_model = X_test_encoded.copy()

typeofFeatures_model = [type_map[col] for col in X_model.columns]

print("Model feature count:", X_model.shape[1])
print("Model features:", X_model.columns.tolist())

Mean baseline score: 0.6481237303227068


100%|██████████| 19/19 [02:17<00:00,  7.22s/it]


RFE summary (feature_count, selected_features, mean_cv_score):
  1: 1 features, 0.737147
  2: 2 features, 0.737147
  3: 3 features, 0.732315
  4: 4 features, 0.737720
  5: 5 features, 0.748074
  6: 6 features, 0.749569
  7: 7 features, 0.764064
  8: 8 features, 0.781549
  9: 9 features, 0.786151
  10: 10 features, 0.792478
  11: 11 features, 0.789026
  12: 12 features, 0.756820
  13: 13 features, 0.762571
  14: 14 features, 0.648584
  15: 15 features, 0.652955
  16: 16 features, 0.647089
  17: 17 features, 0.656981
  18: 18 features, 0.645822
  19: 19 features, 0.648124
Best RFE feature count selected: 10 (min_selected_features=1)
Original dataset shape: (8693, 19)
New dataset shape after feature selection: (8693, 10)
Input features before selection: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'PassengerGroup', 'PassengerNumber', 'CabinDeck', 'CabinNum', 'CabinSide', 'LastName', 'TotalSpend', 'NoSpend', 'SpendPer

In [8]:
# Second DataChecker pass: split the RFE-selected dataset into holdout + repeated folds.
model_checker = DataChecker(
    X=X_model,
    y=y,
    X_test=X_test_model,
    mode="class",
    typeofFeatures=typeofFeatures_model,
)
model_checker.varify_data_types()
model_checker.apply_transformations(use_target_encoder=False)

kfold_splits = model_checker.get_folds(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)
print("KFold splits created successfully.")

No raw text/object columns detected; categorical columns appear to be already encoded.
Categorical features: ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinSide']
Numerical features: ['FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'NoSpend']
No numerical features found, skipping numerical transformations.
No categorical features found, skipping target encoding.
Repeated KFold splits with holdout created successfully!
Number of CV splits per target: 15
Training set size for first fold: 5215
Validation set size for first fold: 2608
Holdout set size: 870
Total original data size: 8693
Target encoding applied to 4 categorical columns across 15/15 folds.
Encoded categorical columns: ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinSide']
KFold splits created successfully.


In [9]:
def make_calibrated_ridge():
    try:
        return CalibratedClassifierCV(estimator=RidgeClassifier(), cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=RidgeClassifier(), cv=3)


def make_all_model_zoo():
    # These are the classification equivalents of the L1_model_zoo families:
    # Ridge, Linear, Lasso, SVR, KNN, RF, XGB, LGBM, CatBoost, MLP, ExtraTrees.
    return {
        "Ridge": make_calibrated_ridge(),
        "Linear": LogisticRegression(max_iter=2000, n_jobs=-1, random_state=RANDOM_STATE),
        "Lasso": LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
        "SVR": SVC(probability=True, cache_size=1000, random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(),
        "RF": RandomForestClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "XGB": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        ),
        "LGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            verbose=-1,
            random_state=RANDOM_STATE,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            allow_writing_files=False,
            verbose=False,
            random_state=RANDOM_STATE,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(256,),
            max_iter=500,
            random_state=RANDOM_STATE,
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }

In [10]:
def fit_model(model, tr, va):
    try:
        model.fit(tr["X"], tr["y"], eval_set=[(va["X"], va["y"])], verbose=False)
    except TypeError:
        try:
            model.fit(tr["X"], tr["y"], eval_set=[(va["X"], va["y"])])
        except TypeError:
            model.fit(tr["X"], tr["y"])
    except Exception:
        model.fit(tr["X"], tr["y"])


def train_all_model_zoo(kfold_splits):
    trained_models = {}
    scores = {}
    train_splits = kfold_splits["train_splits"][0]
    valid_splits = kfold_splits["valid_splits"][0]

    for model_name, template_model in make_all_model_zoo().items():
        print(f"Training {model_name}...")
        fold_models = []
        fold_scores = []
        for tr, va in tqdm(list(zip(train_splits, valid_splits)), desc=model_name):
            model = copy.deepcopy(template_model)
            fit_model(model, tr, va)
            predictions = model.predict(va["X"])
            fold_models.append(model)
            fold_scores.append(accuracy_score(va["y"].to_numpy(), predictions))

        trained_models[model_name] = fold_models
        scores[model_name] = float(np.mean(fold_scores))
        print(f"{model_name} CV accuracy: {scores[model_name]}")

    return trained_models, scores

In [11]:
all_trained_models, cv_scores = train_all_model_zoo(kfold_splits)
cv_scores

Training Ridge...


Ridge: 100%|██████████| 15/15 [00:01<00:00, 10.97it/s]


Ridge CV accuracy: 0.741070819243142
Training Linear...


Linear:   0%|          | 0/15 [00:00<?, ?it/s]c:\Users\ricky\anaconda3\envs\imbd-helper\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
Linear:   7%|▋         | 1/15 [00:00<00:01,  7.25it/s]c:\Users\ricky\anaconda3\envs\imbd-helper\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
Linear:  13%|█▎        | 2/15 [00:00<00:01,  7.04it/s]c:\Users\ricky\anaconda3\envs\imbd-helper\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
Linear:  20%|██

Linear CV accuracy: 0.7410453647290643
Training Lasso...


...

MLP: 100%|██████████| 15/15 [01:34<00:00,  6.30s/it]


MLP CV accuracy: 0.7877538979921133
Training ExtraTrees...


ExtraTrees: 100%|██████████| 15/15 [00:25<00:00,  1.68s/it]


ExtraTrees CV accuracy: 0.7871409305448677


{'Ridge': 0.741070819243142,
 'Linear': 0.7410453647290643,
 'Lasso': 0.7404062367873815,
 'SVR': 0.7515531862070265,
 'KNN': 0.7587371639043227,
 'RF': 0.7925863041379079,
 'XGB': 0.7992587108171095,
 'LGBM': 0.7980317169520789,
 'CatBoost': 0.8007670084396029,
 'MLP': 0.7877538979921133,
 'ExtraTrees': 0.7871409305448677}

In [12]:
kfold_predictor = KFoldPredictor(
    kfold_splits=kfold_splits,
    models=all_trained_models,
    types_of_features=typeofFeatures_model,
    mode="class",
)
holdout_accuracy = kfold_predictor.fit_holdout()
holdout_accuracy

Accuracy for Ridge: 0.7494252873563219
Accuracy for Linear: 0.7517241379310344
Accuracy for Lasso: 0.7494252873563219
Accuracy for SVR: 0.7666666666666667
Accuracy for KNN: 0.8011494252873563
Accuracy for RF: 0.8103448275862069
Accuracy for XGB: 0.8137931034482758
Accuracy for LGBM: 0.8068965517241379
Accuracy for CatBoost: 0.8126436781609195
Accuracy for MLP: 0.7977011494252874
Accuracy for ExtraTrees: 0.7977011494252874
Weights calculated: [0.08656399 0.08682953 0.08656399 0.0885555  0.0925385  0.09360064
 0.09399894 0.09320234 0.09386617 0.0921402  0.0921402 ]
Final accuracy: 0.7919540229885057


0.7919540229885057

In [13]:
test_pred = kfold_predictor.predict(model_checker.X_test)

submission = sample_submission.copy()
submission[TARGET] = test_pred.astype(bool)
submission_path = OUTPUT_DIR / "submission_imbd_helper_all_models_rfe.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
submission.head()

Saved: outputs\spaceship-titanic\submission_imbd_helper_all_models_rfe.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False
